In [1]:
import pandas as pd
import sys, os
from pathlib import Path

import os
import math
sys.path.append(str(Path(os.getcwd()).parent))
from utils.wrappers import measure_time_and_space, measure_time
from typing import Iterable, List
from utils.data_structures import RollingMeanArray

### Getting list of filepaths

In [2]:
@measure_time_and_space
def select_target_csvs(directory):
    # Define target filenames (use a set for O(1) lookup)
    target_files = {
        "MSFT.csv",
        "NVDA.csv",
        "AAPL.csv",
        "GOOGL.csv",
        "AMZN.csv",
        "META.csv",
        "TSLA.csv",
    }

    selected = []
    with os.scandir(directory) as entries:
        for entry in entries:
            if entry.is_file() and entry.name in target_files:
                selected.append(entry.path)
    return selected

@measure_time_and_space
def select_target_csvs_nonrecursive(directory):
    # Define target filenames (use a set for O(1) lookup)
    target_files = [
        "MSFT.csv",
        "NVDA.csv",
        "AAPL.csv",
        "GOOGL.csv",
        "AMZN.csv",
        "META.csv",
        "TSLA.csv",
        ]
    

    selected_paths = []

    # Efficiently iterate through directory (not recursive)
    for filename in os.listdir(directory):
        if filename in target_files:
            selected_paths.append(os.path.join(directory, filename))

    return selected_paths

In [3]:
select_target_csvs(Path.cwd() / "csv")
select_target_csvs_nonrecursive(Path.cwd() / "csv")



[select_target_csvs] Time elapsed: 0.001384 seconds
[select_target_csvs] Peak memory: 3.58 KB
[select_target_csvs_nonrecursive] Time elapsed: 0.000566 seconds
[select_target_csvs_nonrecursive] Peak memory: 38.22 KB


['c:\\Users\\Admin\\.Projects\\Programming Fundamentals\\SIT_ICT1002\\data\\csv\\AAPL.csv',
 'c:\\Users\\Admin\\.Projects\\Programming Fundamentals\\SIT_ICT1002\\data\\csv\\AMZN.csv',
 'c:\\Users\\Admin\\.Projects\\Programming Fundamentals\\SIT_ICT1002\\data\\csv\\GOOGL.csv',
 'c:\\Users\\Admin\\.Projects\\Programming Fundamentals\\SIT_ICT1002\\data\\csv\\META.csv',
 'c:\\Users\\Admin\\.Projects\\Programming Fundamentals\\SIT_ICT1002\\data\\csv\\MSFT.csv',
 'c:\\Users\\Admin\\.Projects\\Programming Fundamentals\\SIT_ICT1002\\data\\csv\\NVDA.csv',
 'c:\\Users\\Admin\\.Projects\\Programming Fundamentals\\SIT_ICT1002\\data\\csv\\TSLA.csv']

In [4]:
file_paths = select_target_csvs(Path.cwd() / "csv")

[select_target_csvs] Time elapsed: 0.001569 seconds
[select_target_csvs] Peak memory: 3.47 KB


### Reading the selected csv files in a efficient manner

In [5]:
@measure_time_and_space

def read_filtered_csvs(file_paths, chunksize=100000):
    """
    Reads multiple CSV files in chunks, filters rows with Date.year > 2024,
    and returns a single combined DataFrame.
    """
    combined_chunks = []  # list to store filtered chunks

    for path in file_paths:
        for chunk in pd.read_csv(path, parse_dates=['Date'], chunksize=chunksize, usecols=lambda col: col != 'Adj Close'):
            filtered_chunk = chunk[chunk['Date'].dt.year > 2024] # Filter rows where year > 2024
            if not filtered_chunk.empty:
                combined_chunks.append(filtered_chunk)

    # Concatenate all filtered chunks into a single DataFrame
    combined_df = pd.concat(combined_chunks, ignore_index=True) if combined_chunks else pd.DataFrame()
    return combined_df

combined_df = read_filtered_csvs(file_paths)

[read_filtered_csvs] Time elapsed: 0.137874 seconds
[read_filtered_csvs] Peak memory: 4505.82 KB


In [6]:
combined_df.head()

,Date,Ticker,Open,High,Low,Close,Volume
0,2025-01-02,AAPL,248.929993,249.100006,241.820007,243.850006,55740700
1,2025-01-02,AAPL,248.929993,249.100006,241.820007,243.850006,55740700
2,2025-01-03,AAPL,243.360001,244.179993,241.889999,243.360001,40244100
3,2025-01-03,AAPL,243.360001,244.179993,241.889999,243.360001,40244100
4,2025-01-06,AAPL,244.309998,247.330002,243.199997,245.000000,45045600


In [7]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2730 entries, 0 to 2729
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    2730 non-null   datetime64[ns]
 1   Ticker  2730 non-null   object        
 2   Open    2730 non-null   float64       
 3   High    2730 non-null   float64       
 4   Low     2730 non-null   float64       
 5   Close   2730 non-null   float64       
 6   Volume  2730 non-null   int64         
dtypes: datetime64[ns](1), float64(4), int64(1), object(1)
memory usage: 149.4+ KB


In [8]:
# Convert date datatype from object to datetime
combined_df = combined_df[combined_df["Date"] < "2025-09-01"]
print("Data ranges from", combined_df["Date"].min(), "to", combined_df["Date"].max())


Data ranges from 2025-01-02 00:00:00 to 2025-08-29 00:00:00


In [9]:
from utils.data_structures import RollingMeanArray

In [10]:
@measure_time_and_space
def sma_pandas(df, window=20):
    """
    Calculate Simple Moving Average (SMA) for the 'Close' column.
    """
    return df['Close'].rolling(window=window).mean()

combined_df["SMA_30"] = sma_pandas(combined_df, window=30)

[sma_pandas] Time elapsed: 0.000436 seconds
[sma_pandas] Peak memory: 58.78 KB


In [21]:
rma = RollingMeanArray(combined_df['Close'],30)

In [25]:
@measure_time_and_space
def sma_rma():
    """
    Calculate Simple Moving Average (SMA) for the 'Close' column using RollingMeanArray class.
    """
    return pd.Series(rma.rolling_mean())
combined_df["SMA_30"] = sma_rma()

[sma_rma] Time elapsed: 0.003081 seconds
[sma_rma] Peak memory: 201.88 KB


In [26]:
@measure_time_and_space
def sma_naive():
    """
    Calculate Simple Moving Average (SMA) for the 'Close' column using RollingMeanArray.naive_rolling_mean method, as a benchmark against the sma_rma method.
    """
    return pd.Series(rma.naive_rolling_mean())
combined_df["SMA_30"] = sma_naive()

[sma_naive] Time elapsed: 0.007709 seconds
[sma_naive] Peak memory: 201.93 KB


In [29]:
rma.window = 90
combined_df["SMA_90"] = sma_rma()
rma.window = 180
combined_df["SMA_180"] = sma_rma()

[sma_rma] Time elapsed: 0.003092 seconds
[sma_rma] Peak memory: 200.50 KB
[sma_rma] Time elapsed: 0.002835 seconds
[sma_rma] Peak memory: 198.34 KB


In [9]:
combined_df.to_csv("mag7_stocks.csv")